In [1]:
import os
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

load_dotenv()

#1 loading (가정)
raw_text = """[스파르타대학교 수강신청 및 성적 장학 규정]
제1조(신청 학점): 학생은 학기당 최소 12학점에서 최대 19학점까지 수강 신청할 수 있습니다.
제2조(초과 신청): 직전 학기 평점 평균이 4.0 이상인 학생은 다음 학기에 최대 21학점까지 신청 가능합니다.
제3조(성적 장학금): 직전 학기 15학점 이상을 이수하고 평점 평균 3.5 이상을 취득한 학생 중 선발합니다.
단, F 학점이 포함된 과목이 있는 경우 성적 장학금 지급 대상에서 제외됩니다."""

#2 chunking (document 객체 리스트로 분할)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=90,
    chunk_overlap=20
)

split_docs = text_splitter.create_documents([raw_text])

print(f"총 {len(split_docs)}개의 청크로 분할되었습니다.\n")

for i, doc in enumerate(split_docs, 1):
    print(f"[청크 {i}]: {doc.page_content}\n")

#3 embedding & storing
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(
    documents=split_docs,
    embedding=embeddings
)

print("인덱싱 완료: 문서가 임베딩되어 벡터 저장소에 안전하게 보관되었습니다.")

총 4개의 청크로 분할되었습니다.

[청크 1]: [스파르타대학교 수강신청 및 성적 장학 규정]
제1조(신청 학점): 학생은 학기당 최소 12학점에서 최대 19학점까지 수강 신청할 수 있습니다.

[청크 2]: 제2조(초과 신청): 직전 학기 평점 평균이 4.0 이상인 학생은 다음 학기에 최대 21학점까지 신청 가능합니다.

[청크 3]: 제3조(성적 장학금): 직전 학기 15학점 이상을 이수하고 평점 평균 3.5 이상을 취득한 학생 중 선발합니다.

[청크 4]: 단, F 학점이 포함된 과목이 있는 경우 성적 장학금 지급 대상에서 제외됩니다.

인덱싱 완료: 문서가 임베딩되어 벡터 저장소에 안전하게 보관되었습니다.


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

user_query = "직전 학기 성적이 4.1점인데 다음 학기에 최대 몇 학점까지 신청할 수 있나요?"

#4 Retrieval
retriever = vector_store.as_retriever(search_kwargs={"k":1})
retrieved_results = retriever.invoke(user_query)

context_chunk = retrieved_results[0].page_context
print("=== [4단계: 검색된 근거 청크] ===")
print(context_chunk)

#5 Generation

